# Lab 04 · Tools: let the model read and change live data

**Prepared by Mustafa Assaf** · Lecture 2: Chain, call tools, act · 15–20 minute lab

## Give the model hands and eyes

**So far:** every fact the model used came from its training or from text we put into the prompt: the policy in Lab 01, the retrieved policies in Lab 02, the earlier replies in Lab 03.

**New:** *tools*. A tool is a function that the model may ask our code to run. A *read* tool looks up live data, such as which hotels have free rooms tonight. A *write* tool changes data, such as booking a room. The model only *asks* for a tool; our code checks the request, runs the function, and returns the result.

We build this up in small steps, on a new example: a hotel booking system.

1. Ask the model about tonight's rooms without a tool: it can only guess, or say that it cannot know.
2. Write a search tool and call it ourselves.
3. Register the tools, and check every request the model makes.
4. Describe the tools to the model, let it pick one, then answer from the tool's result.
5. Add a tool that books a room, so the data changes, and make it wait for the user's confirmation.

**At a glance**

![Lab 04 at a glance: Request, then Pick a tool ({"tool", "args"}), then call_tool() (name · types), then User confirms (write tools only), then search_hotels() (reads live data) or book_room() (changes data), then Answer (from the result).](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/04.svg)

## Pattern · One-Step Tool Agent

*Single-agent pattern from the [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide. Related: Incremental Tool Agent, Registry & Adapter.*

| | |
| --- | --- |
| **Context** | Many tasks need a deterministic capability, such as a lookup, before an answer can be trusted. |
| **Problem** | Free-form model output is hard to execute, while a long reasoning chain is unnecessary for a straightforward tool call. |
| **Solution** | Ask the model for its plan, a tool name, and parameters as JSON; execute the tool; then use the result. |
| **Use when** | One tool is enough per step; latency matters; tool schemas are simple and stable. |
| **Trade-off** | Fast and compact, but tool choice and parameters are made in one shot. The Incremental Tool Agent separates them, at a higher cost. |
| **Flow** | Goal → Plan + choose → Tool → Synthesize |
| **Quality attributes** | Benefits: Latency, Cost, Performance, Maintainability. Liabilities: Reliability, Accuracy, Security, Observability. |
| **In this lab** | The model picks `search_hotels` or `book_room` and its arguments as JSON; our code checks the name and the argument types, runs the tool, and the model answers from the result. A write tool waits for the user's confirmation, which addresses the pattern's security liability. |

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Practice answers are scripted. For a live model, choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`), and change `MODE` to `"live"`. `ask` stops after 12 calls; run this cell again to reset the count.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit']):
    %pip install -q "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/9c78f2181b0f793c60b2b081c79ae74276026e7d.zip"
from coursekit import make_ask
MODE = "practice"
PROVIDER = "freeinference"  # or "gemini" or "deepseek"
ask = make_ask(mode=MODE, provider=PROVIDER)

## Step 1 · Without a tool, the model cannot know

`hotels` is the booking system's live data. It changes every minute, and no model was trained on it. We ask the model whether Harbor View has a room tonight, then look at the data ourselves. In practice mode, the scripted reply is a confident guess. In our live tests, Qwen and Gemini instead said that they cannot see live availability: more honest, but just as unable to answer.

In [ ]:
import json

hotels = {  # the booking system's live data: rooms change every minute
    "Harbor View": {"city": "Lisbon", "price": 140, "rooms_left": 0, "status": "open"},
    "Alfama Inn": {"city": "Lisbon", "price": 95, "rooms_left": 3, "status": "open"},
    "Tagus Lodge": {"city": "Lisbon", "price": 80, "rooms_left": 5, "status": "closed for renovation"},
    "Canal House": {"city": "Amsterdam", "price": 160, "rooms_left": 2, "status": "open"},
}
question = "Does the Harbor View hotel in Lisbon have a free room tonight, and what does it cost?"
guess = ask(question, example="Yes, Harbor View usually has rooms; a double costs about 120 euros a night.")
print("Model alone:   ", guess)
print("Booking system:", hotels["Harbor View"])

## Step 2 · A search tool

`search_hotels` is a plain Python function: it returns the open hotels in a city that have a free room, cheapest first, within a price limit. We call it ourselves first, before any model is involved. Its answer is always as fresh as the data.

In [ ]:
def search_hotels(city: str, max_price: int = 1000) -> list:
    """Open hotels in a city with at least one free room, cheapest first."""
    found = [{"hotel": name, "price": h["price"], "rooms_left": h["rooms_left"]}
             for name, h in hotels.items()
             if h["city"].lower() == city.lower() and h["status"] == "open"
             and h["rooms_left"] > 0 and h["price"] <= max_price]
    return sorted(found, key=lambda hotel: hotel["price"])

print(search_hotels("Lisbon"))
print(search_hotels("Lisbon", max_price=90))

## Step 3 · Register the tools, and check every request

`call_tool` is the boundary between the model and our system. It runs a tool only if the reply is JSON, the name is known, and every argument has the right type. Every failure comes back as an error that the model can read, never as a crash: a reply that is not JSON, or a call with a missing argument, raises an error that the `except` line turns into a message. It checks `type(v) is` rather than `isinstance`, because Python treats `True` as an `int`. A tool marked `writes` also needs the user's confirmation; Step 5 uses it.

In [ ]:
TOOLS = {"search_hotels": {"run": search_hotels, "args": {"city": str, "max_price": int}, "writes": False,
                           "description": "Find open hotels with a free room in a city, under a price."}}

def call_tool(reply, confirmed=False):
    """Run the tool the model asked for, after checking its name, its arguments, and the user's consent."""
    try:
        request = json.loads(reply)
        spec = TOOLS.get(request.get("tool"))
        if spec is None:
            return {"error": f"unknown tool {request.get('tool')!r}"}
        args = request.get("args", {})
        if any(a not in spec["args"] or type(v) is not spec["args"][a] for a, v in args.items()):
            return {"error": f"bad arguments {args}"}
        if spec["writes"] and not confirmed:
            return {"error": "this tool changes data and needs the user's confirmation"}
        return spec["run"](**args)
    except (json.JSONDecodeError, TypeError) as error:  # not JSON, or a missing argument
        return {"error": f"bad request: {error}"}

## Step 4 · Describe the tools; the model picks one, then answers

The model never sees our code. It sees each tool's name, description, and argument types, which `describe` writes out from `TOOLS`. Then come two model calls. The first gets the tool descriptions and the request, and replies with JSON naming a tool and its arguments. Our code runs it with `call_tool`. The second call gets the tool's result and answers in one sentence, using only that result.

In [ ]:
def describe():
    return "\n".join(f"- {name}({', '.join(f'{a}: {t.__name__}' for a, t in spec['args'].items())}): "
                     f"{spec['description']}" for name, spec in TOOLS.items())

FORMAT = 'Reply with JSON only, no code fences: {"tool": "<name>", "args": {...}}'
request = "Find me a hotel in Lisbon for under 120 euros a night."
choice = ask(f"Tools:\n{describe()}\nRequest: {request}\n{FORMAT}",
             example='{"tool": "search_hotels", "args": {"city": "Lisbon", "max_price": 120}}')
result = call_tool(choice)
answer = ask(f"Request: {request}\nTool result: {json.dumps(result)}\nAnswer in one sentence, using only the result.",
             example="Alfama Inn has 3 rooms left at 95 euros a night.")
print("Model asked for:", choice, "\nTool returned:  ", result, "\nModel answered: ", answer)

## Step 5 · A tool that changes the data

`book_room` is a *write* tool: it books one room and lowers `rooms_left`. It refuses anything it cannot do, and then nothing changes. The model asks for a booking, but `call_tool` runs it only after the user confirms. Searching again shows the new state: tools always see live data.

In [ ]:
bookings = []

def book_room(hotel: str, guest: str) -> dict:
    """Book one room. It changes the data, so it refuses anything it cannot do."""
    h = hotels.get(hotel)
    if h is None or h["status"] != "open" or h["rooms_left"] < 1:
        return {"error": f"{hotel!r} cannot be booked now"}
    h["rooms_left"] -= 1
    bookings.append({"hotel": hotel, "guest": guest})
    return {"booked": hotel, "rooms_left": h["rooms_left"]}

TOOLS["book_room"] = {"run": book_room, "args": {"hotel": str, "guest": str}, "writes": True,
                      "description": "Book one room in a hotel for a guest. Changes the data."}
choice = ask(f"Tools:\n{describe()}\nRequest: Book the Alfama Inn for Dana.\n{FORMAT}",
             example='{"tool": "book_room", "args": {"hotel": "Alfama Inn", "guest": "Dana"}}')
print("Before confirmation:", call_tool(choice))
print("After confirmation: ", call_tool(choice, confirmed=True))
full = '{"tool": "book_room", "args": {"hotel": "Harbor View", "guest": "Dana"}}'
print("A full hotel:       ", call_tool(full, confirmed=True))
print("Search again:       ", search_hotels("Lisbon"), "| bookings:", bookings)

## What you should see

Step 1 shows the model's reply, a guess or a refusal, next to the real data: Harbor View has no free room and costs 140 euros. Step 2 finds the Alfama Inn in Lisbon; Tagus Lodge is cheaper but closed, so the 90-euro search returns nothing. In Step 4, the model asks for `search_hotels` with Lisbon and 120, and answers from the result. In Step 5, the booking is refused until the user confirms, then Alfama Inn drops to 2 rooms; Harbor View is refused because it is full, and nothing changes.

## Try it

1. Mark the Alfama Inn as `"closed for renovation"` and run Steps 2 and 4 again. The tool now returns `[]`, and no model was retrained. (The practice answer is scripted; in live mode, the answer changes too.)
2. Make the model ask for a tool that does not exist, for example `"delete_hotel"`, in Step 4's scripted reply. What does `call_tool` return?
3. Send `"max_price": "120"` (a string) instead of `120`, then `"args": {}` with no city. Which check catches each one?
4. Add a `cancel_booking(hotel, guest)` write tool. It should give the room back, and refuse a booking that does not exist.
5. In live mode, does the model pick the right tool for "Is anything free in Amsterdam?" and for "Book Canal House for Sam"?

## Takeaway

A model knows only what it was trained on and what we put into the prompt. Tools give it live data and let it act, but our code stays in charge: it publishes what each tool does, checks every request, returns errors as data, and asks the user before anything changes. Lab 05 lets the model call tools again and again, choosing each next step from what it has seen.

## Read further

- [Building effective agents: the augmented LLM](https://www.anthropic.com/research/building-effective-agents)
- [Function calling with the Chat Completions API](https://developers.openai.com/api/docs/guides/function-calling)